# Confidential Federated Compute Workshop

## Secure Data Processing with Trusted Execution Environments

---

### Welcome!

In this interactive workshop, you'll learn how **Confidential Federated Compute** protects sensitive data while enabling secure collaborative analysis. We'll explore:

- 🔒 **How data stays encrypted** even during processing
- 🛡️ **Trusted Execution Environments (TEEs)** that guarantee security
- 🔑 **Key Management Systems (KMS)** that control data access
- 📋 **Access Policies** that define who can process your data
- ⚡ **Cryptographic integrity** that prevents unauthorized access

**No prior security knowledge required!** We'll explain everything step by step.

---

### What You'll Do Today

1. ✅ Connect to the workshop server
2. ✅ Learn about TEEs, KMS, and encryption
3. ✅ Execute a complete secure data processing flow
4. ✅ See what happens when security policies don't match (cryptographic failure!)
5. ✅ Understand how this protects your data in the real world

Let's get started!

---

## Part 1: Setup & Connection

### Prerequisites

Before running this workshop, make sure the workshop server is running:

```bash
# Start the workshop server (in a separate terminal)
bazel run //containers/workshop-server:workshop_server -- --quiet
```

The `--quiet` flag suppresses VM boot logs for a cleaner experience.

**Server Configuration:**
- Maximum concurrent sessions: **40**
- KMS pool size: **5 pre-started instances**
- Each session is automatically assigned a KMS from the pool

---

### Step 1: Import Required Libraries

We'll use Python's `requests` library to communicate with the workshop server.

In [ ]:
import requests
import time
import random
import json
from typing import Optional, Dict, Any, List

# Server configuration
SERVER_HOST = "localhost"
SERVER_PORT = 3000
BASE_URL = f"http://{SERVER_HOST}:{SERVER_PORT}"

print("✅ Libraries imported successfully!")
print(f"📡 Server URL: {BASE_URL}")

### Step 2: Define Helper Functions

These functions will help us interact with the workshop server. Don't worry about the details - we'll explain what each one does as we use it!

In [ ]:
class WorkshopClient:
    """Client for interacting with the Workshop Server."""
    
    def __init__(self, base_url: str):
        self.base_url = base_url
        self.session_id = None
        
    def get_status(self) -> dict:
        """Get server status and session capacity."""
        response = requests.get(f"{self.base_url}/status")
        response.raise_for_status()
        return response.json()
    
    def create_session(self, timeout_seconds: int = 3600) -> dict:
        """Create a new session (KMS auto-assigned from pool)."""
        response = requests.post(
            f"{self.base_url}/sessions",
            json={"timeout_seconds": timeout_seconds}
        )
        response.raise_for_status()
        data = response.json()
        self.session_id = data["session_id"]
        return data
    
    def start_tee(self, memory_size: str = "2G") -> dict:
        """Start a TEE service for this session."""
        response = requests.post(
            f"{self.base_url}/sessions/{self.session_id}/tee",
            json={"memory_size": memory_size}
        )
        response.raise_for_status()
        return response.json()
    
    def get_tee_evidence(self) -> dict:
        """Get attestation evidence from the TEE."""
        response = requests.get(
            f"{self.base_url}/sessions/{self.session_id}/tee/evidence"
        )
        response.raise_for_status()
        return response.json()
    
    def get_tee_reference_values(self) -> dict:
        """Get reference values (binary hashes) from TEE evidence."""
        response = requests.get(
            f"{self.base_url}/sessions/{self.session_id}/tee/reference_values"
        )
        response.raise_for_status()
        return response.json()
    
    def rotate_keyset(self, keyset_id: int, ttl_seconds: int) -> dict:
        """Rotate KMS keyset to generate new encryption keys."""
        response = requests.post(
            f"{self.base_url}/sessions/{self.session_id}/kms/rotate_keyset",
            json={"keyset_id": keyset_id, "ttl_seconds": ttl_seconds}
        )
        response.raise_for_status()
        return response.json()
    
    def create_variant_policy(self, variant_name: str, src_nodes: List[int], dst_nodes: List[int]) -> dict:
        """Create a pipeline variant policy."""
        response = requests.post(
            f"{self.base_url}/sessions/{self.session_id}/policies/variant",
            json={
                "variant_policy_name": variant_name,
                "src_node_ids": src_nodes,
                "dst_node_ids": dst_nodes
            }
        )
        response.raise_for_status()
        return response.json()
    
    def create_data_access_policy(self, policy_name: str, pipeline_name: str, variant_names: List[str]) -> dict:
        """Create a data access policy."""
        response = requests.post(
            f"{self.base_url}/sessions/{self.session_id}/policies/data_access",
            json={
                "policy_name": policy_name,
                "logical_pipeline_name": pipeline_name,
                "variant_policy_names": variant_names
            }
        )
        response.raise_for_status()
        return response.json()
    
    def derive_keys(self, policy_name: str) -> dict:
        """Derive encryption keys from a data access policy."""
        response = requests.post(
            f"{self.base_url}/sessions/{self.session_id}/kms/derive_keys",
            json={"policy_name": policy_name}
        )
        response.raise_for_status()
        return response.json()
    
    def encrypt_data(self, blob_name: str, plaintext: str, policy_name: str, key_index: int = 0) -> dict:
        """Encrypt data using HPKE."""
        response = requests.post(
            f"{self.base_url}/sessions/{self.session_id}/encrypt",
            json={
                "blob_name": blob_name,
                "plaintext": plaintext,
                "policy_name": policy_name,
                "public_key_index": key_index
            }
        )
        response.raise_for_status()
        return response.json()
    
    def register_pipeline(self, invocation_name: str, pipeline_name: str, 
                         policy_name: str, keyset_id: int, ttl: int) -> dict:
        """Register a pipeline invocation with KMS."""
        response = requests.post(
            f"{self.base_url}/sessions/{self.session_id}/kms/register_pipeline",
            json={
                "invocation_name": invocation_name,
                "logical_pipeline_name": pipeline_name,
                "data_access_policy_name": policy_name,
                "keyset_id": keyset_id,
                "ttl_seconds": ttl
            }
        )
        response.raise_for_status()
        return response.json()
    
    def authorize_transform(self, invocation_name: str) -> dict:
        """Authorize a pipeline transform (TEE gets decryption keys)."""
        response = requests.post(
            f"{self.base_url}/sessions/{self.session_id}/kms/authorize_transform",
            json={"invocation_name": invocation_name}
        )
        response.raise_for_status()
        return response.json()
    
    def process_with_tee(self, blob_names: List[str], invocation_name: str) -> dict:
        """Send encrypted data to TEE for processing."""
        response = requests.post(
            f"{self.base_url}/sessions/{self.session_id}/tee/process",
            json={
                "blob_names": blob_names,
                "invocation_name": invocation_name
            }
        )
        response.raise_for_status()
        return response.json()
    
    def delete_session(self) -> None:
        """Delete the session and release KMS back to pool."""
        if self.session_id:
            response = requests.delete(f"{self.base_url}/sessions/{self.session_id}")
            response.raise_for_status()
            self.session_id = None

# Create the client instance
client = WorkshopClient(BASE_URL)

print("✅ Helper functions defined!")
print("📝 We're ready to connect to the server.")

### Step 3: Test Server Connection

Let's make sure the workshop server is running and check its current capacity.

In [ ]:
# Check server status
try:
    status = client.get_status()
    print("✅ Server is online!")
    print(f"\n📊 Server Status:")
    print(f"   Status: {status['status']}")
    if 'sessions' in status:
        sessions = status['sessions']
        print(f"   Active Sessions: {sessions['active']}/{sessions['max']}")
        print(f"   Available Slots: {sessions['available']}")
        print(f"   At Capacity: {'Yes' if sessions['at_limit'] else 'No'}")
    print(f"\n💡 The server has a pool of 5 KMS instances shared across all sessions.")
except Exception as e:
    print(f"❌ Could not connect to server: {e}")
    print(f"\n⚠️  Make sure the workshop server is running:")
    print(f"   bazel run //containers/workshop-server:workshop_server -- --quiet")

---

## Part 2: Understanding the Concepts

Before we start processing data, let's understand the key concepts behind Confidential Federated Compute.

---

### 🛡️ What is a Trusted Execution Environment (TEE)?

A **TEE** is like a **secure vault inside a computer**. It's a special protected area where:

- ✅ Code runs in **complete isolation** (even the operating system can't peek inside)
- ✅ The hardware **cryptographically proves** what code is running
- ✅ Data is **encrypted** before entering and after leaving
- ✅ No one - not even server administrators - can access the data inside

**Real-world analogy:** Imagine sending a locked box to a secure facility. The facility has a special room where:
1. Only authorized robots can enter
2. The robots are certified and verified
3. Everything they do is recorded
4. They can open your box, process the contents, and lock the results in a new box
5. No humans ever see your data!

**Examples of TEE technology:**
- AMD SEV-SNP (what we're using today)
- Intel TDX
- ARM TrustZone

---

### 🔍 Why Do We Need TEEs? The Problem of Trust

Imagine you want to analyze sensitive data (medical records, financial data, personal information) using cloud computing. **How do you know the cloud provider won't peek at your data?**

**The Traditional Problem:**
```
Your Data → Upload to Cloud → ❓ What happens here? ❓
                                 |
                                 ├─ Cloud admins can access it?
                                 ├─ Malicious software running?
                                 ├─ Data being copied/logged?
                                 └─ No way to verify!
```

**You have to blindly trust:**
- ❌ The cloud provider won't access your data
- ❌ Their employees are trustworthy
- ❌ Their systems are secure
- ❌ No bugs or vulnerabilities exist
- ❌ They're running the code they claim

This is called the **"trust problem"** - and it's a huge barrier for sensitive data processing!

---

#### 🛡️ The Solution: TEEs + Attestation

TEEs solve this problem through **hardware-based attestation**. Instead of trusting people or companies, you trust the **laws of physics and mathematics**!

**How It Works:**

```
Step 1: Boot the TEE
  → Hardware measures every byte of code
  → Creates cryptographic hash (SHA-384)

Step 2: Generate Attestation Evidence
  → Hardware signs measurements with secret key
  → Evidence = tamper-proof cryptographic proof

Step 3: Provide Endorsements
  → Manufacturer provides certificate chain
  → Proves hardware is genuine

Step 4: Extract Reference Values
  → Derive binary hashes from evidence
  → Used in access policies to authorize TEEs
```

---

#### 🌳 Oak Containers: The TEE Framework

In this workshop, we use **Oak Containers** - a framework built by Google for running applications in TEEs.

**What Oak Containers Provide:**

1. **🔐 Evidence Generation**
   - Automatic collection of hardware attestation
   - Cryptographic measurements of all loaded code
   - Signed by the TEE hardware (AMD SEV-SNP)
   - Accessible via API: `get_evidence()`

2. **📜 Endorsements**
   - Certificate chains from hardware manufacturer
   - Proves the TEE hardware is genuine
   - Links evidence to trusted root of trust
   - Accessible via API: `get_endorsements()`

3. **🎯 Reference Values**
   - Binary hashes extracted from evidence
   - Identifies the specific software version
   - Used in access policies to authorize TEEs
   - Derived via: `evidence_to_reference_values()`

**The Oak Attestation Flow:**

```
Oak Container (Your TEE)
        ↓
Oak Orchestrator (Manages TEE lifecycle)
        ↓
AMD SEV-SNP Hardware (Signs evidence)
        ↓
Your Client (Verifies and extracts reference values)
```

---

#### 🔑 Three Key Artifacts

When you interact with a TEE, you get three critical pieces of information:

**1. Evidence (Attestation Report)**
- **What it is:** Cryptographic proof of what's running in the TEE
- **Contains:** Software hashes, hardware identity, measurements
- **Signed by:** TEE hardware's secret key (unforgeable!)
- **Size:** ~1-4 KB of binary data
- **In this workshop:** You'll fetch it in Step 2a

**2. Endorsements (Certificate Chain)**
- **What it is:** Proof that the hardware is genuine
- **Contains:** Certificates from AMD/Intel, public keys
- **Signed by:** Hardware manufacturer
- **Purpose:** Links evidence to trusted root
- **Size:** ~10-50 KB of binary data
- **In this workshop:** You'll fetch it in Step 2a

**3. Reference Values (Binary Hashes)**
- **What it is:** "Fingerprint" identifying the exact software
- **Contains:** SHA-384 hashes of kernel, system, application
- **Derived from:** Evidence (via attestation verification)
- **Purpose:** Used in access policies to authorize specific TEEs
- **Size:** ~200-500 bytes
- **In this workshop:** You'll extract them in Step 2b

---

#### 🎯 Why This Matters for Security

**Without Attestation:**
```
Client: "Hey cloud, run this code on my data"
Cloud:  "Sure, trust me!"
Client: "How do I know you're actually running that code?"
Cloud:  "Just trust me!"
        ↑ Blind trust required ❌
```

**With TEE Attestation:**
```
Client: "Show me proof of what you're running"
TEE:    "Here's evidence signed by hardware: 0x3a4f2b..."
Client: "Let me verify... ✓ Hardware signature valid"
Client: "Let me check hashes... ✓ Running approved code"
Client: "Let me verify endorsements... ✓ Genuine hardware"
Client: "OK, here's my encrypted data!"
        ↑ Cryptographic proof, no trust needed! ✅
```

**The Security Guarantees:**
- ✅ **Verifiable Execution** - Prove exactly what code is running
- ✅ **Hardware Root of Trust** - Based on silicon, not software
- ✅ **Tamper Detection** - Any code change breaks the signature
- ✅ **No Admin Access** - Even cloud admins can't peek inside
- ✅ **Auditable** - Anyone can verify the attestation

---

#### 💡 In Today's Workshop

You'll see this in action when we:

1. **Start a TEE** → Oak Container boots in AMD SEV-SNP
2. **Fetch Evidence** → Get cryptographic proof from hardware
3. **Get Endorsements** → Verify hardware authenticity
4. **Extract Reference Values** → Get binary hashes for policies
5. **Create Access Policy** → Include reference values
6. **KMS Verifies TEE** → Checks evidence before releasing keys

**This is how we achieve "trust through cryptography, not through promises!"** 🎉

---

### 🚀 What Can TEEs Do? Real-World Applications!

Now that you understand what a TEE is, let's explore the **exciting applications** you can build with them!

#### Transformations & Data Processing Pipelines

A **transformation** is a data processing step that runs inside a TEE. It:
- ✅ Takes encrypted data as input
- ✅ Decrypts it **only inside the TEE** (never in the open!)
- ✅ Processes it according to its purpose
- ✅ Encrypts the output before returning it
- ✅ Can pass results to the next transformation

A **data processing pipeline** chains multiple transformations together:

```
Input Data → [Transform 1 in TEE] → [Transform 2 in TEE] → Output
 (encrypted)      (decrypt/process)     (decrypt/process)    (encrypted)
```

**Key Characteristics:**
- 🔒 **End-to-End Encryption** - Data encrypted at rest and in transit
- 🛡️ **Secure Computation** - Processing happens in hardware-protected TEEs
- 🔗 **Composable** - Chain multiple transformations together
- 📋 **Policy-Governed** - Access policies control which TEEs can process data
- 🎯 **Verifiable** - Attestation proves what code is running

---

#### 🌟 Real-World Use Cases - The Applications Are Unlimited!

##### 1. 📊 **SQL Analytics on Sensitive Data**

Run SQL queries on encrypted databases without exposing the data:

**Example: Healthcare Data Analysis**
- Multiple hospitals encrypt patient records
- SQL query runs inside a TEE
- Results aggregated and encrypted
- Individual patient data never exposed!

**Use Cases:**
- Cross-organizational analytics (healthcare, finance)
- Regulatory compliance (GDPR, HIPAA)
- Government statistics and census
- Multi-party data analysis

---

##### 2. 🤖 **AI & Machine Learning Workloads**

Train machine learning models on encrypted private data:

**Example: Collaborative AI Training**
- Mobile devices encrypt training data
- Neural network training happens in TEE
- Model updates without exposing individual data
- Privacy-preserving federated learning!

**Use Cases:**
- Healthcare AI (medical imaging, diagnosis)
- Financial fraud detection
- Personalized recommendations
- Natural language processing on private text
- Computer vision on sensitive images

---

##### 3. 🧬 **Federated Learning & Scientific Research**

Enable collaborative research without sharing raw data:

**Example: Drug Discovery**
- Pharmaceutical companies share encrypted clinical trial data
- Aggregation TEE combines statistics
- Differential privacy adds protection
- Breakthrough discoveries without data exposure!

**Use Cases:**
- Medical research across institutions
- Genomics analysis (privacy-critical)
- Climate modeling with sensitive data
- Social science studies

---

##### 4. 🔍 **Other Exciting Applications**

The possibilities are endless! TEEs enable:

- **💰 Financial Services**
  - Cross-bank fraud detection
  - Secure trading algorithms
  - Risk assessment across institutions

- **🏛️ Government & Public Services**
  - Census data analysis
  - Tax fraud detection
  - Public health monitoring

- **🛍️ E-Commerce & Advertising**
  - Privacy-preserving recommendations
  - Secure bidding systems
  - Customer analytics without leakage

- **🌐 Blockchain & Web3**
  - Private smart contracts
  - Confidential transactions
  - Secure multi-party computation

---

#### 📋 How Policies Control Pipelines

Remember our access policy structure? It defines the **exact pipeline** your data flows through:

```
Access Policy = Blueprint for Data Processing

Transform 1 "Anonymize"  →  Transform 2 "Aggregate"
  - TEE Hash: 0x3a4f2b...    - TEE Hash: 0x8c1d9e...
  - Budget: 1x                - Budget: 1x
  - Node 0 → Node 1           - Node 1 → Node 2
```

**This ensures:**
- 🎯 **Predictability** - You know exactly what will happen to your data
- 🔒 **Security** - Only authorized TEEs can access data
- 📝 **Auditability** - The pipeline is publicly verifiable
- 🛡️ **Immutability** - Cannot be modified after data upload

---

#### 💡 Why This Matters

Traditional cloud computing requires you to **trust** the cloud provider:
- ❌ Admins can access your data
- ❌ Software bugs can leak information
- ❌ Hard to verify what code is running
- ❌ Compliance challenges

**With TEEs + Confidential Federated Compute:**
- ✅ **Hardware-enforced** security (don't need to trust admins!)
- ✅ **Cryptographically verified** code execution
- ✅ **Policy-bound** data processing
- ✅ **Privacy-preserving** collaborative analysis

**You can finally answer questions like:**
- "What's the average age of patients with this condition?" *(across multiple hospitals)*
- "Can we detect fraud patterns?" *(across competing banks)*
- "What's the best treatment?" *(using private clinical trial data)*

**All without exposing the underlying private data!** 🎉

---

In today's workshop, we're using a simple **test_concat** TEE that concatenates strings. But the same principles apply to:
- 📊 Complex SQL queries (GROUP BY, JOIN, aggregate functions)
- 🤖 Deep learning models (PyTorch, TensorFlow)
- 🧬 Scientific computations (statistical analysis, simulations)
- 💰 Financial algorithms (risk models, trading strategies)

**The sky's the limit!** 🚀

---

### 🔑 What is the Key Management Service (KMS)?

The **KMS** is the **gatekeeper** for encryption keys. It:

- 🔐 **Generates encryption keys** for protecting data
- 🔍 **Verifies TEE identity** before giving out decryption keys
- 📋 **Enforces access policies** (only authorized TEEs can decrypt data)
- ⏰ **Manages key expiration** (crypto-erasure for data TTL)
- 🛡️ **Runs inside a TEE itself** (for maximum security)

**How it works:**
```
Client                    KMS                     TEE
   │                       │                       │
   │  1. Give me a key     │                       │
   │──────────────────────>│                       │
   │                       │                       │
   │  2. Here's public key │                       │
   │<──────────────────────│                       │
   │                       │                       │
   │  3. Encrypt data      │                       │
   │  with public key      │                       │
   │                       │                       │
   │  4. Upload encrypted  │                       │
   │  data to storage      │                       │
   │                       │                       │
   │                       │  5. I'm authorized!  │
   │                       │    (shows evidence)   │
   │                       │<──────────────────────│
   │                       │                       │
   │                       │  6. Verified! Here's  │
   │                       │    decryption key     │
   │                       │──────────────────────>│
   │                       │                       │
   │                       │              7. Decrypt & process
```

**Key insight:** The KMS **never** gives decryption keys to untrusted code. Only TEEs with the correct "reference values" (binary hashes) can decrypt your data.

---

### 📋 What are Access Policies?

An **access policy** is like a **contract** that defines:

1. **Who** can access the data (specific TEE binaries, identified by hash)
2. **What** processing steps are allowed (pipeline transforms)
3. **How many times** the data can be accessed (access budget)
4. **What constraints** apply (e.g., differential privacy parameters)

**Policy Structure (simplified):**
```
DataAccessPolicy
├── Pipeline: "my_analysis_pipeline"
│   ├── Variant: "version_1.0"
│   │   ├── Transform 1: square numbers
│   │   │   ├── Input: node 0 (raw data)
│   │   │   ├── Output: node 1 (squared data)
│   │   │   ├── Reference Values: hash_of_square_binary
│   │   │   └── Access Budget: 1 time
│   │   │
│   │   └── Transform 2: sum results
│   │       ├── Input: node 1 (squared data)
│   │       ├── Output: node 2 (final sum)
│   │       ├── Reference Values: hash_of_sum_binary
│   │       └── Access Budget: 1 time
```

**Why this matters:**
- ✅ Policies are **bound to data at encryption time** (via hash)
- ✅ Cannot be changed after data is uploaded
- ✅ Publicly auditable (published in transparency log)
- ✅ Enforced cryptographically (wrong policy = wrong decryption key)

---

### 🔐 How Does HPKE Encryption Work?

**HPKE** (Hybrid Public Key Encryption) is the encryption algorithm that protects your data throughout the entire flow. Let's trace how keys are generated, used, and passed through each step!

---

#### 🔑 The Complete Key Journey Through All Steps

**Step 1: Keyset Rotation**
```
Action: client.rotate_keyset(keyset_id, ttl_seconds)

What happens in KMS:
  1. KMS generates Input Key Material (IKM) - a random secret
  2. Stores: { keyset_id: 1234, ikm: <secret>, ttl: 7200s }
  3. This IKM will be used to derive ALL encryption keys

You get: ✓ Keyset rotated confirmation
Keys generated: None yet (just the seed material)
```

**Step 2: Fetch TEE Evidence & Reference Values**
```
Action: client.get_tee_evidence() + client.get_tee_reference_values()

What happens:
  1. TEE provides cryptographic proof (evidence)
  2. Extract binary hashes (reference values)
  3. These will identify which TEE can decrypt

You get: Evidence + Endorsements + Reference Values
Keys generated: None (just identity proof)
```

**Step 3: Create Access Policy + Derive Keys**
```
Action: client.create_data_access_policy() → client.derive_keys()

What happens in KMS:
  1. Policy created with TEE reference values
  2. Compute: policy_hash = SHA256(policy)
  3. Derive HPKE keypair:
     
     key_material = HKDF-SHA256(
         ikm=keyset.ikm,           ← From Step 1
         salt=keyset_id,
         info=policy_hash           ← Binds key to policy!
     )
     
     (public_key, private_key) = X25519_KeyGen(key_material)
     
  4. KMS keeps private_key secret
  5. Returns public_key to client

You get: public_key (for encryption)
KMS stores: private_key (for later decryption)
```

**🔍 Key Insight:** The `policy_hash` is used as the "info" parameter in HKDF. This means:
- Different policies → Different hashes → Different keys
- Policy is cryptographically bound to the encryption!

---

**Step 4: Client Encrypts Data**
```
Action: client.encrypt_data(plaintext, policy_name)

Client-side HPKE encryption:
  1. Get public_key from Step 3
  
  2. Generate ephemeral keypair (one-time use):
     (eph_public, eph_private) = X25519_KeyGen(random)
  
  3. Perform ECDH key exchange:
     shared_secret = X25519_DH(
         eph_private,    ← Client's ephemeral private key
         public_key      ← KMS public key from Step 3
     )
  
  4. Derive encryption key:
     enc_key = HKDF-SHA256(shared_secret)
  
  5. Encrypt with AES-128-GCM:
     ciphertext = AES_GCM_Encrypt(
         key=enc_key,
         plaintext=chunk_data,
         nonce=fixed  ← Safe because key is one-time
     )
  
  6. Package encrypted blob:
     blob = {
       metadata: {
         encapsulated_public_key: eph_public,  ← TEE needs this!
         total_size: len(plaintext)
       },
       data: ciphertext
     }

You send: Encrypted blob (with ephemeral public key)
Client forgets: eph_private (discarded after encryption)
```

**📦 What's in the encrypted blob:**
- `eph_public` - Ephemeral public key (needed for TEE to derive shared secret)
- `ciphertext` - Your data, encrypted with AES-128-GCM
- `auth_tag` - Ensures data hasn't been tampered with

---

**Step 6-7: Register Pipeline & Authorize Transform**
```
Action: client.register_pipeline() → client.authorize_transform()

What happens in KMS:
  1. Register pipeline with policy
  
  2. Authorize transform:
     - KMS receives TEE evidence
     - Verifies evidence matches policy reference values ✓
     - Derives private_key (same HKDF as Step 3)
     - Derives decryption keys for all blobs
  
  3. Encrypt keys with TEE's public key:
     protected_response = Encrypt(
         tee_public_key,  ← From TEE evidence
         private_key      ← KMS private key from Step 3
     )
     
  4. Send protected_response to TEE (via untrusted pipeline)

You get: protected_response (encrypted for TEE only)
Contains: KMS private_key (encrypted with TEE's public key)
```

**🔐 Security Note:** The `protected_response` travels through untrusted code (your client), but only the TEE can decrypt it!

---

**Step 8: TEE Processes Encrypted Data**
```
Action: client.process_with_tee(blob_names, invocation_name)

What happens inside TEE:
  1. TEE decrypts protected_response:
     private_key = Decrypt(
         tee_private_key,      ← TEE's secret key
         protected_response    ← From Step 7
     )
     
     Now TEE has the KMS private_key!
  
  2. For each encrypted blob:
     a) Extract eph_public from blob metadata
     
     b) Perform ECDH (reverse of Step 4):
        shared_secret = X25519_DH(
            private_key,   ← KMS private key (from protected_response)
            eph_public     ← From blob metadata
        )
        
        This derives the SAME shared_secret as Step 4!
     
     c) Derive decryption key:
        enc_key = HKDF-SHA256(shared_secret)
        
        This derives the SAME enc_key as Step 4!
     
     d) Decrypt with AES-128-GCM:
        plaintext = AES_GCM_Decrypt(
            key=enc_key,
            ciphertext=blob.data
        )
  
  3. Process plaintext data (e.g., concatenate chunks)
  
  4. Return results (encrypted or plaintext, depending on TEE)

TEE outputs: Processed results
TEE forgets: All keys and plaintext (memory cleared)
```

---

#### 🎯 The Complete Flow Visualization

```
┌─────────────────────────────────────────────────────────────┐
│                    HPKE Key Flow                            │
└─────────────────────────────────────────────────────────────┘

Step 1: KMS Keyset Rotation
  KMS: Generate IKM (secret seed)

Step 3: Derive Keys from Policy
  KMS: (public_key, private_key) = HKDF(IKM, policy_hash)
       ↓
       Returns public_key to client
       Keeps private_key secret

Step 4: Client Encrypts
  Client: Generate (eph_public, eph_private)
          shared_secret = DH(eph_private, public_key)
          enc_key = HKDF(shared_secret)
          ciphertext = AES(enc_key, plaintext)
          Send: {eph_public, ciphertext}

Step 7: KMS Authorizes TEE
  KMS: Verify TEE evidence ✓
       Encrypt private_key with TEE's public key
       Send: protected_response

Step 8: TEE Decrypts & Processes
  TEE: private_key = Decrypt(protected_response)
       shared_secret = DH(private_key, eph_public)
       enc_key = HKDF(shared_secret)  ← SAME as Step 4!
       plaintext = AES_Decrypt(enc_key, ciphertext)
       Process plaintext
```

---

#### 🔒 Why This Is Secure

**1. Policy Binding (Step 3)**
```
Keys are derived with policy_hash:
  Policy A → Hash A → Keys A
  Policy B → Hash B → Keys B

Keys A ≠ Keys B → Cannot decrypt with wrong policy!
```

**2. Ephemeral Keys (Step 4)**
```
Each encryption uses a fresh ephemeral keypair:
  - eph_private discarded immediately
  - Even if one ciphertext is broken, others remain safe
  - Forward secrecy guaranteed
```

**3. Protected Response (Step 7)**
```
KMS private_key encrypted with TEE public key:
  - Travels through untrusted client
  - Only authorized TEE can decrypt
  - Untrusted code never sees decryption keys!
```

**4. TEE Isolation (Step 8)**
```
All decryption happens inside TEE:
  - Hardware-enforced isolation
  - Plaintext never leaves TEE
  - Keys cleared from memory after use
```

---

#### 💡 Key Insights

**Three Types of Keys:**
1. **IKM (Input Key Material)** - Secret seed in KMS (Step 1)
2. **HPKE Keypair** - Derived from IKM + policy_hash (Step 3)
3. **Ephemeral Keypair** - One-time keys per encryption (Step 4)

**Two Critical Exchanges:**
1. **Client ↔ KMS** - Public key shared in Step 3
2. **KMS ↔ TEE** - Private key in protected_response (Step 7)

**The Magic of ECDH:**
```
Client computes:  DH(eph_private, kms_public)  = shared_secret
TEE computes:     DH(kms_private, eph_public)  = shared_secret
                                                   ↑
                                        SAME SECRET!
                                        
Both derive the same encryption key, but:
- Client never sees kms_private
- TEE never sees eph_private
- Untrusted code never sees either!
```

**Algorithms Used:**
- **Key Exchange:** X25519 (Elliptic Curve Diffie-Hellman)
- **Key Derivation:** HKDF-SHA256
- **Encryption:** AES-128-GCM (authenticated encryption)

---

You'll see all of this in action in the hands-on section! Pay attention to:
- Step 3: The public key you receive
- Step 4: The encrypted blob you create
- Step 7: The protected_response from KMS
- Step 8: TEE processing without exposing plaintext

**This is how we achieve end-to-end encryption with TEE-based processing!** 🎉

---

---

## Part 3: Normal KMS Flow (9 Steps)

Now let's execute the complete Confidential Federated Compute flow! We'll follow the 9-step encryption lifecycle from the KMS documentation.

**Overview of the 9 Steps:**
1. Rotate keyset (generate new encryption keys)
2. Start TEE service and get attestation
3. Create access policy with TEE reference values
4. Derive encryption keys from policy
5. Client encrypts data with HPKE
6. Store encrypted data
7. Register pipeline invocation with KMS
8. Authorize transform (TEE gets decryption keys)
9. TEE processes encrypted data

---

### Step 0: Create a Session

First, we need to create a session. The server will automatically assign us a KMS from the pool of 5 instances (using a least-loaded algorithm).

In [ ]:
# Create a new session
print("🔄 Creating a new session...")
session_data = client.create_session(timeout_seconds=3600)

print(f"\n✅ Session created successfully!")
print(f"   Session ID: {session_data['session_id']}")
print(f"   Timeout: {session_data['timeout_seconds']} seconds")

if 'kms_service' in session_data:
    kms = session_data['kms_service']
    print(f"\n🔐 KMS Auto-Assigned from Pool:")
    print(f"   Address: {kms['address']}:{kms['port']}")
    print(f"   Auto-assigned: {kms.get('auto_assigned', False)}")

print(f"\n💡 This session will automatically release the KMS back to the pool when we're done.")

### Step 1: Rotate Keyset

The KMS maintains **keysets** - collections of encryption keys with expiration times. When we rotate a keyset, we:
- Add a new key to the keyset
- Mark it as the active key for encryption
- Set a TTL (time-to-live) for the key

Old keys remain available for decrypting older data, but all new encryptions use the latest key.

In [ ]:
# Generate a unique keyset ID
keyset_id = random.randint(1000, 9999)
ttl_seconds = 7200  # 2 hours

print(f"🔄 Rotating keyset {keyset_id}...")
rotate_result = client.rotate_keyset(keyset_id, ttl_seconds)

print(f"\n✅ Keyset rotated successfully!")
print(f"   Keyset ID: {keyset_id}")
print(f"   TTL: {ttl_seconds} seconds ({ttl_seconds // 3600} hours)")
print(f"\n💡 The KMS has generated new key material for deriving encryption keys.")
print(f"   Any data encrypted with this keyset will expire after {ttl_seconds // 3600} hours.")

### Step 2: Start TEE Service and Get Attestation

Now we'll start a **Trusted Execution Environment** (TEE) that will process our encrypted data. The TEE will provide:

- **Evidence**: Cryptographic proof of what code is running inside the TEE
- **Endorsements**: Signed certificates from the hardware manufacturer
- **Reference Values**: Binary hashes that identify the specific software

This attestation data allows the KMS to verify the TEE before giving it decryption keys.

In [ ]:
# Start the TEE service
print("🚀 Starting TEE service...")
print("   This will launch a secure VM running test_concat...")
print("   (This may take 30-60 seconds)\n")

tee_result = client.start_tee(memory_size="2G")

print(f"✅ TEE service started successfully!")
if 'instances' in tee_result and len(tee_result['instances']) > 0:
    instance = tee_result['instances'][0]
    print(f"   Instance ID: {instance['instance_id']}")
    print(f"   Address: {instance['address']}:{instance['port']}")

# Wait a moment for TEE to fully initialize
print(f"\n⏳ Waiting 5 seconds for TEE to fully initialize...")
time.sleep(5)

print(f"\n💡 The TEE is now running in complete isolation inside a secure enclave.")

### Step 2a: Fetch TEE Evidence (Attestation)

Let's retrieve the attestation evidence from the TEE. This is the cryptographic proof that the TEE is genuine and running the correct software.

In [ ]:
# Get TEE evidence
print("🔍 Fetching TEE attestation evidence...")
evidence_data = client.get_tee_evidence()

if 'evidence' in evidence_data:
    evidence_hex = evidence_data['evidence']
    evidence_size = len(evidence_hex) // 2  # Hex string is 2 chars per byte
    
    print(f"\n✅ Attestation evidence retrieved!")
    print(f"   Evidence size: {evidence_size} bytes")
    print(f"   Evidence (first 64 chars): {evidence_hex[:64]}...")
    print(f"\n💡 This evidence contains:")
    print(f"   - Hardware measurements (CPU, firmware)")
    print(f"   - Software measurements (kernel, application)")
    print(f"   - Cryptographic signatures from AMD SEV-SNP")
else:
    print(f"⚠️  No evidence available (might be using mock/test mode)")

if 'endorsements' in evidence_data:
    endorsements_hex = evidence_data['endorsements']
    endorsements_size = len(endorsements_hex) // 2
    print(f"\n✅ Endorsements retrieved!")
    print(f"   Endorsements size: {endorsements_size} bytes")
    print(f"   Endorsements (first 64 chars): {endorsements_hex[:64]}...")
else:
    print(f"\n⚠️  No endorsements available")

### Step 2b: Get TEE Reference Values

From the attestation evidence, we can extract **reference values** - these are the binary hashes that uniquely identify the TEE software. We'll use these in our access policy to ensure only THIS specific TEE can decrypt our data.

In [ ]:
# Get reference values
print("🔍 Extracting reference values from evidence...")
ref_values_data = client.get_tee_reference_values()

if 'reference_values' in ref_values_data:
    ref_values_hex = ref_values_data['reference_values']
    ref_values_size = len(ref_values_hex) // 2
    
    print(f"\n✅ Reference values extracted!")
    print(f"   Size: {ref_values_size} bytes")
    print(f"   Reference values (first 64 chars): {ref_values_hex[:64]}...")
    print(f"\n💡 These reference values are the 'fingerprint' of our TEE.")
    print(f"   Only TEEs with matching fingerprints can decrypt our data!")
else:
    print(f"\n⚠️  No reference values available")

### Step 3: Create Access Policy with TEE Reference Values

Now we'll create an **access policy** that specifies:
1. A **variant policy** defining the data flow (input node 0 → output node 1)
2. A **data access policy** that includes our TEE's reference values

The policy will be **bound to the encryption keys** via its hash (SHA-256).

In [ ]:
# Create variant policy
variant_policy_name = "test_concat_variant"
print(f"📋 Creating variant policy: {variant_policy_name}")
print(f"   Data flow: node 0 (client uploads) → node 1 (TEE output)")

variant_result = client.create_variant_policy(
    variant_name=variant_policy_name,
    src_nodes=[0],  # Read from client uploads
    dst_nodes=[1]   # Write to output
)
print(f"✅ Variant policy created!")

# Create data access policy
policy_name = "test_concat_access_policy"
pipeline_name = "test_concat_pipeline"
print(f"\n📋 Creating data access policy: {policy_name}")
print(f"   Pipeline: {pipeline_name}")
print(f"   This policy includes the TEE reference values we extracted earlier.")

policy_result = client.create_data_access_policy(
    policy_name=policy_name,
    pipeline_name=pipeline_name,
    variant_names=[variant_policy_name]
)
print(f"✅ Data access policy created!")

print(f"\n💡 The KMS will now use SHA-256(policy) as the 'info' parameter in HKDF.")
print(f"   This binds the encryption keys to this specific policy.")

### Step 4: Derive Encryption Keys from Policy

Now we ask the KMS to derive HPKE encryption keys for our policy. The KMS will:
1. Compute policy_hash = SHA-256(policy)
2. Derive HPKE key pair using HKDF(keyset_ikm, info=policy_hash)
3. Return the public key for client encryption

In [ ]:
# Derive encryption keys
print(f"🔐 Deriving encryption keys for policy: {policy_name}")
keys_result = client.derive_keys(policy_name=policy_name)

print(f"\n✅ Encryption keys derived!")
if 'public_keys' in keys_result and len(keys_result['public_keys']) > 0:
    public_key = keys_result['public_keys'][0]
    print(f"   Number of keys: {len(keys_result['public_keys'])}")
    print(f"   Public key (first 32 chars): {public_key[:32]}...")
    print(f"\n💡 We'll use this public key to encrypt our data with HPKE.")
    print(f"   Only the KMS can derive the corresponding private key.")
    print(f"   The KMS will only give the private key to TEEs with matching reference values!")

### Step 5: Client Encrypts Data

Now we'll encrypt some data using HPKE! We'll encrypt two separate chunks:
- Chunk 1: `"Hello from chunk 1!"`
- Chunk 2: `"Hello from chunk 2!"`

The `test_concat` TEE will concatenate these chunks together.

In [ ]:
# Encrypt first chunk
chunk1_plaintext = "Hello from chunk 1!"
blob_name_1 = "chunk_1"

print(f"🔒 Encrypting chunk 1...")
print(f"   Plaintext: '{chunk1_plaintext}'")
print(f"   Size: {len(chunk1_plaintext)} bytes")

encrypt1_result = client.encrypt_data(
    blob_name=blob_name_1,
    plaintext=chunk1_plaintext,
    policy_name=policy_name,
    key_index=0
)

if 'encrypted_blob' in encrypt1_result:
    encrypted1_hex = encrypt1_result['encrypted_blob']
    encrypted1_size = len(encrypted1_hex) // 2
    print(f"✅ Chunk 1 encrypted!")
    print(f"   Encrypted size: {encrypted1_size} bytes")
    print(f"   First 64 chars: {encrypted1_hex[:64]}...")

# Encrypt second chunk
chunk2_plaintext = "Hello from chunk 2!"
blob_name_2 = "chunk_2"

print(f"\n🔒 Encrypting chunk 2...")
print(f"   Plaintext: '{chunk2_plaintext}'")
print(f"   Size: {len(chunk2_plaintext)} bytes")

encrypt2_result = client.encrypt_data(
    blob_name=blob_name_2,
    plaintext=chunk2_plaintext,
    policy_name=policy_name,
    key_index=0
)

if 'encrypted_blob' in encrypt2_result:
    encrypted2_hex = encrypt2_result['encrypted_blob']
    encrypted2_size = len(encrypted2_hex) // 2
    print(f"✅ Chunk 2 encrypted!")
    print(f"   Encrypted size: {encrypted2_size} bytes")
    print(f"   First 64 chars: {encrypted2_hex[:64]}...")

print(f"\n💡 Both chunks are now encrypted with HPKE.")
print(f"   The encrypted data includes:")
print(f"   - Ephemeral public key (for ECDH key exchange)")
print(f"   - Encrypted content (with AES-128-GCM)")
print(f"   - Authentication tag (prevents tampering)")

### Step 6: Store Encrypted Data & Register Pipeline

The encrypted data is now stored in our session. Next, we register a **pipeline invocation** with the KMS. This tells the KMS:
- We're about to run a pipeline
- Which policy governs this execution
- What the TTL is for intermediate results

The KMS returns an **invocation ID** that we'll use when authorizing transforms.

In [ ]:
# Register pipeline invocation
invocation_name = "test_concat_invocation"
intermediates_ttl = 3600  # 1 hour

print(f"📝 Registering pipeline invocation: {invocation_name}")
print(f"   Pipeline: {pipeline_name}")
print(f"   Policy: {policy_name}")
print(f"   Keyset ID: {keyset_id}")
print(f"   Intermediates TTL: {intermediates_ttl} seconds")

register_result = client.register_pipeline(
    invocation_name=invocation_name,
    pipeline_name=pipeline_name,
    policy_name=policy_name,
    keyset_id=keyset_id,
    ttl=intermediates_ttl
)

print(f"\n✅ Pipeline invocation registered!")
if 'invocation_id' in register_result:
    print(f"   Invocation ID: {register_result['invocation_id']}")

print(f"\n💡 The KMS has created pipeline-specific key material for intermediate results.")

### Step 7: Authorize Transform (TEE Gets Decryption Keys)

Now comes the critical security check! We call `authorize_transform` which:

1. Sends the TEE's evidence to the KMS
2. KMS verifies the evidence matches the policy's reference values
3. If verified, KMS derives decryption keys and encrypts them with the TEE's public key
4. Returns a **ProtectedResponse** (encrypted keys) to the TEE

This ensures **only authorized TEEs can decrypt the data**!

In [ ]:
# Authorize transform
print(f"🔐 Authorizing transform for invocation: {invocation_name}")
print(f"   The KMS will verify our TEE's evidence...")

authorize_result = client.authorize_transform(invocation_name=invocation_name)

print(f"\n✅ Transform authorized!")
print(f"   The KMS has verified the TEE matches our policy's reference values.")
print(f"   The TEE has received encrypted decryption keys.")

if 'protected_response' in authorize_result:
    protected_response = authorize_result['protected_response']
    print(f"   Protected response size: {len(protected_response) // 2} bytes")
    print(f"\n💡 The protected response contains:")
    print(f"   - Decryption keys for our encrypted data")
    print(f"   - Encrypted with the TEE's public key")
    print(f"   - Only this TEE can decrypt it!")

### Step 8: TEE Processes Encrypted Data

Finally, we send the encrypted data to the TEE for processing! The TEE will:

1. Decrypt the ProtectedResponse using its private key → get decryption keys
2. Decrypt both encrypted chunks using HPKE
3. Concatenate the plaintext chunks
4. Return the result

**Note:** In a production system, the result would be encrypted. For this demo, `test_concat` returns plaintext for simplicity.

In [ ]:
# Send encrypted data to TEE for processing
print(f"⚡ Sending encrypted data to TEE...")
print(f"   Blobs: {blob_name_1}, {blob_name_2}")
print(f"   The TEE will:")
print(f"   1. Decrypt the ProtectedResponse to get decryption keys")
print(f"   2. Decrypt both chunks with HPKE")
print(f"   3. Concatenate the plaintext")
print(f"   4. Return the result\n")

process_result = client.process_with_tee(
    blob_names=[blob_name_1, blob_name_2],
    invocation_name=invocation_name
)

if 'encrypted_results' in process_result:
    result_hex = process_result['encrypted_results']
    result_bytes = bytes.fromhex(result_hex)
    result_text = result_bytes.decode('utf-8', errors='replace')
    
    print(f"✅ TEE processing complete!")
    print(f"   Result size: {len(result_bytes)} bytes")
    print(f"   Result: '{result_text}'")
    
    print(f"\n💡 Expected: '{chunk1_plaintext}{chunk2_plaintext}'")
    
    expected = chunk1_plaintext + chunk2_plaintext
    if result_text == expected:
        print(f"   ✅ SUCCESS! The TEE correctly concatenated both chunks!")
    else:
        print(f"   ⚠️  Result doesn't match expected output")
else:
    print(f"❌ No results returned from TEE")

### Step 9: Summary of Normal Flow

Congratulations! You've successfully completed the entire Confidential Federated Compute flow!

**What we accomplished:**

✅ **Created a session** with auto-assigned KMS from pool  
✅ **Rotated keyset** to generate fresh encryption keys  
✅ **Started a TEE** running in a secure enclave  
✅ **Retrieved attestation** evidence and reference values  
✅ **Created access policy** binding the TEE's identity  
✅ **Derived encryption keys** tied to the policy hash  
✅ **Encrypted data** with HPKE using X25519 + AES-128-GCM  
✅ **Registered pipeline** invocation with KMS  
✅ **Authorized transform** (TEE received decryption keys)  
✅ **Processed encrypted data** entirely inside the TEE  

**Key Security Properties:**

🔒 Data was **never** exposed in plaintext outside the TEE  
🔑 Only the authorized TEE could decrypt the data  
📋 The policy was cryptographically bound to the encryption  
🛡️ Hardware attestation proved the TEE's identity  
⏰ Keys will automatically expire after their TTL  

---

---

## Part 4: Security Demo - Policy Hash Mismatch

Now let's see what happens when we try to **trick the system** by using mismatched policies!

**The Attack Scenario:**
1. We'll create **Policy A** and encrypt data with its key
2. Then create **Policy B** (different pipeline name = different hash)
3. Try to authorize the TEE with Policy B
4. Send the data (encrypted with Policy A's key) to the TEE
5. Watch the decryption **fail** due to key mismatch!

**Why this matters:**
This demonstrates that the policy is **cryptographically bound** to the encryption. You cannot change policies after data is encrypted - the HKDF key derivation ensures different policies produce different keys.

---

### Important: Relaunch TEE for Clean State

To properly demonstrate the mismatch, we need to **delete the current session** (which releases the TEE) and create a new one with a fresh TEE.

In [ ]:
# Clean up the previous session
print("🧹 Cleaning up previous session...")
client.delete_session()
print("✅ Previous session deleted (KMS released back to pool)")

# Create a new session for the mismatch test
print("\n🔄 Creating new session for mismatch test...")
session_data = client.create_session(timeout_seconds=3600)
print(f"✅ New session created: {session_data['session_id']}")

if 'kms_service' in session_data:
    kms = session_data['kms_service']
    print(f"🔐 KMS assigned: {kms['address']}:{kms['port']}")

# Use the same keyset ID (or rotate a new one)
mismatch_keyset_id = random.randint(1000, 9999)
print(f"\n🔄 Rotating new keyset {mismatch_keyset_id}...")
client.rotate_keyset(mismatch_keyset_id, 7200)
print(f"✅ Keyset rotated")

# Start a fresh TEE
print(f"\n🚀 Starting fresh TEE service...")
print(f"   (This may take 30-60 seconds)\n")
tee_result = client.start_tee(memory_size="2G")
print(f"✅ TEE started!")

# Wait for TEE initialization
print(f"\n⏳ Waiting 5 seconds for TEE initialization...")
time.sleep(5)
print(f"✅ Ready for mismatch test!")

### Create Policy A (For Encryption)

First, we'll create **Policy A** with pipeline name `"pipeline_a_name"`. We'll use this policy to derive encryption keys and encrypt our data.

In [ ]:
# Create Policy A variant
policy_a_variant = "policy_a_variant"
policy_a_name = "policy_a_access_policy"
pipeline_a_name = "pipeline_a_name"  # ← Note the pipeline name!

print(f"📋 Creating Policy A...")
print(f"   Variant: {policy_a_variant}")
print(f"   Policy: {policy_a_name}")
print(f"   Pipeline: {pipeline_a_name}")

client.create_variant_policy(policy_a_variant, [0], [1])
client.create_data_access_policy(policy_a_name, pipeline_a_name, [policy_a_variant])

print(f"✅ Policy A created!")

# Derive keys from Policy A
print(f"\n🔐 Deriving encryption keys from Policy A...")
keys_result = client.derive_keys(policy_name=policy_a_name)
print(f"✅ Keys derived from Policy A")

if 'public_keys' in keys_result and len(keys_result['public_keys']) > 0:
    key_a = keys_result['public_keys'][0]
    print(f"   Public key A: {key_a[:32]}...")

print(f"\n💡 These keys are derived with: HKDF(keyset_ikm, info=SHA256(Policy A))")

### Create Policy B (For Authorization - DIFFERENT!)

Now we'll create **Policy B** with a **different pipeline name**: `"pipeline_b_name"`. This will result in a different policy hash, and therefore different encryption keys!

In [ ]:
# Create Policy B variant
policy_b_variant = "policy_b_variant"
policy_b_name = "policy_b_access_policy"
pipeline_b_name = "pipeline_b_name"  # ← DIFFERENT pipeline name!

print(f"📋 Creating Policy B (DIFFERENT!)...")
print(f"   Variant: {policy_b_variant}")
print(f"   Policy: {policy_b_name}")
print(f"   Pipeline: {pipeline_b_name} ← DIFFERENT!")

client.create_variant_policy(policy_b_variant, [0], [1])
client.create_data_access_policy(policy_b_name, pipeline_b_name, [policy_b_variant])

print(f"✅ Policy B created!")

print(f"\n⚠️  WARNING: Policy B has a different pipeline name!")
print(f"   SHA256(Policy A) ≠ SHA256(Policy B)")
print(f"   Therefore: HKDF keys for A ≠ HKDF keys for B")

### Encrypt Data with Policy A

Now we'll encrypt some sensitive data using **Policy A's encryption key**.

In [ ]:
# Encrypt data with Policy A key
mismatch_plaintext = "Secret data encrypted with Policy A!"
mismatch_blob = "mismatch_test_blob"

print(f"🔒 Encrypting data with Policy A key...")
print(f"   Plaintext: '{mismatch_plaintext}'")
print(f"   Policy: {policy_a_name}")

encrypt_result = client.encrypt_data(
    blob_name=mismatch_blob,
    plaintext=mismatch_plaintext,
    policy_name=policy_a_name,  # ← Using Policy A!
    key_index=0
)

if 'encrypted_blob' in encrypt_result:
    encrypted_size = len(encrypt_result['encrypted_blob']) // 2
    print(f"\n✅ Data encrypted with Policy A key!")
    print(f"   Encrypted size: {encrypted_size} bytes")
    print(f"   Encryption used: HKDF(keyset_ikm, info=SHA256(Policy A))")

### Register Pipeline with Policy B (MISMATCH!)

Here's where the attack attempt happens! We'll register a pipeline using **Policy B** (even though the data was encrypted with Policy A).

In [ ]:
# Register pipeline with Policy B
mismatch_invocation = "mismatch_invocation"

print(f"📝 Registering pipeline with Policy B...")
print(f"   ⚠️  Data encrypted with: Policy A")
print(f"   ⚠️  Registering with: Policy B (DIFFERENT!)")
print(f"   ⚠️  This is the mismatch!\n")

register_result = client.register_pipeline(
    invocation_name=mismatch_invocation,
    pipeline_name=pipeline_b_name,  # ← Using Policy B!
    policy_name=policy_b_name,      # ← Using Policy B!
    keyset_id=mismatch_keyset_id,
    ttl=3600
)

print(f"✅ Pipeline registered with Policy B")
print(f"\n💡 Note: The KMS does NOT validate policy hash consistency here.")
print(f"   The cryptographic failure will happen during TEE decryption!")

### Authorize Transform with Policy B

The authorization will succeed - the TEE will receive decryption keys derived from **Policy B**.

In [ ]:
# Authorize transform with Policy B
print(f"🔐 Authorizing transform with Policy B...")

authorize_result = client.authorize_transform(invocation_name=mismatch_invocation)

print(f"\n✅ Transform authorized!")
print(f"   The TEE received decryption keys for Policy B")
print(f"   But the data was encrypted with Policy A's key!")
print(f"\n⚠️  Decryption keys: HKDF(keyset_ikm, info=SHA256(Policy B))")
print(f"   Encrypted with: HKDF(keyset_ikm, info=SHA256(Policy A))")
print(f"   These are DIFFERENT keys!")

### Send Mismatched Data to TEE - Watch it Fail!

Now for the moment of truth! We'll send the encrypted data (encrypted with Policy A's key) to the TEE, which has decryption keys for Policy B.

**Expected Result:** Decryption will fail because the keys don't match!

In [ ]:
# Send mismatched data to TEE
print(f"⚡ Sending mismatched data to TEE...")
print(f"   Data encrypted with: Policy A key")
print(f"   TEE has keys for: Policy B")
print(f"   Expected result: DECRYPTION FAILURE\n")

process_result = client.process_with_tee(
    blob_names=[mismatch_blob],
    invocation_name=mismatch_invocation
)

# Check if decryption succeeded or failed
if 'encrypted_results' in process_result and process_result['encrypted_results']:
    # Unexpected success!
    print(f"⚠️  UNEXPECTED SUCCESS!")
    print(f"   The TEE somehow decrypted the data.")
    print(f"   This suggests the policy hashes were actually the same.")
    
    result_hex = process_result['encrypted_results']
    result_bytes = bytes.fromhex(result_hex)
    result_text = result_bytes.decode('utf-8', errors='replace')
    print(f"   Result: {result_text}")
else:
    # Expected failure!
    print(f"✅ DECRYPTION FAILED AS EXPECTED!")
    print(f"\n🎉 This demonstrates cryptographic integrity!")
    print(f"\n   What happened:")
    print(f"   1. TEE received Policy B decryption keys")
    print(f"   2. TEE tried to decrypt data encrypted with Policy A keys")
    print(f"   3. HPKE decryption failed (wrong key)")
    print(f"   4. TEE could not process the data")
    print(f"\n   Security implications:")
    print(f"   ✅ Policies are cryptographically bound to encryption")
    print(f"   ✅ Cannot switch policies after encryption")
    print(f"   ✅ Different policy hash → Different encryption key")
    print(f"   ✅ Wrong key → Decryption failure (safe failure!)")
    print(f"   ✅ No need for KMS to validate policy consistency")
    print(f"   ✅ Cryptography provides the integrity guarantee!")

### Understanding the Cryptographic Failure

Let's visualize why the decryption failed:

```
ENCRYPTION (with Policy A):
┌────────────────────────────────────────────────────┐
│ 1. policy_hash_A = SHA256(Policy A)               │
│ 2. key_material_A = HKDF(keyset_ikm,              │
│                          info=policy_hash_A)      │
│ 3. (pk_A, sk_A) = derive_keypair(key_material_A)  │
│ 4. encrypted_data = HPKE_Encrypt(pk_A, plaintext) │
└────────────────────────────────────────────────────┘

DECRYPTION ATTEMPT (with Policy B keys):
┌────────────────────────────────────────────────────┐
│ 1. policy_hash_B = SHA256(Policy B)               │
│ 2. key_material_B = HKDF(keyset_ikm,              │
│                          info=policy_hash_B)      │
│ 3. (pk_B, sk_B) = derive_keypair(key_material_B)  │
│ 4. Try: HPKE_Decrypt(sk_B, encrypted_data)        │
│    ❌ FAIL! sk_B ≠ sk_A                           │
└────────────────────────────────────────────────────┘

WHY IT FAILS:
- Policy A ≠ Policy B (different pipeline names)
- SHA256(Policy A) ≠ SHA256(Policy B)
- HKDF with different "info" → different output
- key_material_A ≠ key_material_B
- sk_A ≠ sk_B (different private keys!)
- sk_B cannot decrypt data encrypted for pk_A
```

**This is a FEATURE, not a bug!**

The cryptographic binding ensures that:
- You cannot retroactively change which policy governs data access
- The policy hash becomes part of the encryption key derivation
- Any attempt to use a different policy results in cryptographic failure
- No complex validation logic needed - the math enforces the rules!

---

---

## Part 5: Cleanup & Wrap-Up

Let's clean up our session and review what we learned!

---

### Cleanup: Delete Session

When we delete our session:
- The TEE VM is terminated
- The KMS is released back to the pool (available for other sessions)
- All session data is cleared

In [ ]:
# Clean up session
print("🧹 Cleaning up session...")
client.delete_session()
print("✅ Session deleted successfully!")
print("\n💡 The KMS has been released back to the pool.")
print("   It's now available for other sessions to use.")

# Check final server status
print("\n📊 Final Server Status:")
status = client.get_status()
if 'sessions' in status:
    sessions = status['sessions']
    print(f"   Active Sessions: {sessions['active']}/{sessions['max']}")
    print(f"   Available Slots: {sessions['available']}")

---

## 🚀 Production-Ready Features: KMS Scalability & Crypto-Erasure

Beyond the security features we explored today, the KMS is designed for **production workloads** with two critical capabilities:

---

### 📈 KMS Scalability: RAFT Consensus & High Availability

In production, a single KMS instance isn't enough! You need:
- **High Availability** - System stays online even if servers fail
- **Fault Tolerance** - Automatic recovery from failures
- **Consistency** - All KMS instances agree on key state
- **Scalability** - Handle billions of requests per day

#### How KMS Achieves This: RAFT Consensus

**RAFT** is a distributed consensus algorithm that keeps multiple KMS instances synchronized:

```
┌────────────────────────────────────────────────────────┐
│         KMS Cluster (Multiple Instances)               │
└────────────────────────────────────────────────────────┘

  ┌──────────┐      ┌──────────┐      ┌──────────┐
  │  KMS 1   │      │  KMS 2   │      │  KMS 3   │
  │ (Leader) │◄────►│(Follower)│◄────►│(Follower)│
  └─────┬────┘      └──────────┘      └──────────┘
        │
        │ RAFT Replication
        ↓
  ┌──────────┐      ┌──────────┐
  │  KMS 4   │      │  KMS 5   │
  │(Follower)│      │(Follower)│
  └──────────┘      └──────────┘

All instances share:
- Encryption keys (IKM for key derivation)
- Pipeline state (privacy budgets, usage tracking)
- Access policies
- Key expiration times (for crypto-erasure)
```

**How RAFT Works:**

1. **Leader Election**
   - One KMS instance is elected as the **Leader**
   - Leader coordinates all state updates
   - If Leader fails, a new one is automatically elected

2. **Log Replication**
   - Leader receives write requests (rotate keys, update state)
   - Leader replicates changes to **Followers**
   - Changes committed only after **majority** acknowledgment

3. **Read Operations**
   - Any KMS instance can serve read requests
   - Reads are consistent across all instances
   - Load balanced across the cluster

**Example Flow:**

```
Step 1: Client requests key rotation
  Client → KMS Leader: "Rotate keyset 1234"

Step 2: Leader proposes change
  KMS Leader → All Followers: "Append: rotate keyset 1234"

Step 3: Followers acknowledge
  KMS 2: "✓ Appended to my log"
  KMS 3: "✓ Appended to my log"
  KMS 4: "✓ Appended to my log"

Step 4: Leader commits (majority reached)
  KMS Leader: "Committed! Keyset 1234 rotated"

Step 5: All instances apply the change
  All KMS instances now have the new key!
```

**Benefits:**

- ✅ **Fault Tolerance**: Can survive `(N-1)/2` failures (e.g., 5 instances → tolerate 2 failures)
- ✅ **No Single Point of Failure**: If Leader dies, new one elected in seconds
- ✅ **Strong Consistency**: All instances see the same state
- ✅ **Horizontal Scaling**: Add more instances to handle more load
- ✅ **Automatic Recovery**: Failed instances catch up when they rejoin

**In Production:**
- Run 5-7 KMS instances across multiple data centers
- Handle billions of key derivations per day
- 99.99% uptime with automatic failover
- Each instance runs in its own TEE for security

---

### 🔥 Crypto-Erasure: Automatic Data Expiration

One of the most powerful features of the KMS is **crypto-erasure** - the ability to make encrypted data permanently inaccessible by deleting encryption keys.

#### The Problem: Data Retention Policies

Regulations (GDPR, HIPAA, etc.) often require:
- ❌ Data must be deleted after N days
- ❌ "Right to be forgotten" - delete user data on request
- ❌ Traditional deletion is hard (backups, replicas, logs)

**Traditional Approach:**
```
Find all copies of data → Delete each copy → Hope you got them all
                            ↑
                     Difficult and error-prone!
```

#### The Solution: Crypto-Erasure

Instead of deleting the data, **delete the encryption key**:

```
┌──────────────────────────────────────────────────────┐
│  Time T=0: Data encrypted with key K1               │
├──────────────────────────────────────────────────────┤
│  Encrypted Data: 0x7a3f4b2e8c...  (stored forever)  │
│  Encryption Key: K1                (has TTL = 30d)  │
└──────────────────────────────────────────────────────┘
                        ↓
                   30 days pass
                        ↓
┌──────────────────────────────────────────────────────┐
│  Time T=30d: Key K1 expires                         │
├──────────────────────────────────────────────────────┤
│  Encrypted Data: 0x7a3f4b2e8c...  (still exists)    │
│  Encryption Key: [DELETED]         (erased!)        │
│                                                      │
│  Result: Data is now PERMANENTLY inaccessible! 🔒   │
└──────────────────────────────────────────────────────┘
```

**Why This Works:**
- Encrypted data without the key is just random bits
- Computationally infeasible to decrypt (AES-128-GCM security)
- No need to track down all data copies
- Instant, verifiable erasure

---

#### How KMS Implements TTL (Time-To-Live)

**1. Keyset Rotation with TTL**

When you rotate a keyset (as we did in Step 1 of the workshop), you specify a TTL:

```python
client.rotate_keyset(
    keyset_id=1234,
    ttl_seconds=7200  # 2 hours
)
```

The KMS stores:
- **Key Material (IKM)**: Secret for deriving encryption keys
- **Issued At**: Timestamp when key was created
- **Expires At**: Issued At + TTL

**2. Monotonic Time Tracking**

The KMS maintains a **replicated, monotonically increasing** time:

```
KMS Internal Clock:
  - Cannot go backwards (monotonic)
  - Synchronized across all RAFT instances
  - Based on (untrusted) system time
  - Updated on every operation
```

**Why Monotonic?**
- If system time jumps backward → KMS ignores it
- Prevents "time travel" attacks
- Guarantees keys expire eventually

**3. Automatic Key Expiration**

The KMS periodically checks for expired keys:

```
Every N seconds:
  current_time = KMS.get_time()
  
  for each key in keystore:
    if key.expires_at <= current_time:
      ERASE key.material  # Crypto-erasure!
      mark key as expired
```

**4. Cascading Expiration for Pipeline Intermediates**

When you register a pipeline (Step 6), you specify TTL for intermediates:

```python
client.register_pipeline(
    invocation_name="my_pipeline",
    ttl_seconds=3600  # 1 hour for intermediates
)
```

The KMS ensures:
- Input keys must outlive intermediate keys
- Pipeline cannot extend data lifetime beyond input TTL
- All intermediates expire with the pipeline

```
Timeline:
  T=0:   Upload data (key expires at T=30d)
  T=1d:  Start pipeline (intermediates expire at T=2d)
  T=2d:  Intermediate keys deleted
  T=30d: Original data keys deleted
```

---

#### 🎯 Real-World Benefits

**1. Regulatory Compliance Made Easy**

```
GDPR Requirement: "Delete user data after 90 days"

Traditional Approach:
  - Find all databases with user data
  - Find all backups
  - Find all logs and caches
  - Delete from each location
  - Hope you didn't miss anything
  → Complex, error-prone, expensive

With Crypto-Erasure:
  - Set TTL = 90 days when encrypting
  - KMS automatically deletes keys after 90 days
  - All encrypted data becomes inaccessible
  → Simple, automatic, verifiable ✅
```

**2. "Right to be Forgotten"**

```
User: "Delete my data!"

With Crypto-Erasure:
  1. Identify keys associated with this user
  2. Delete those keys immediately
  3. User's encrypted data is now unreadable
  → Instant, cryptographically guaranteed deletion
```

**3. Defense in Depth**

```
Even if encrypted data is:
  - Copied to unauthorized storage
  - Leaked in a backup
  - Stolen by an attacker

After key expiration:
  → Data is just random bits (unrecoverable)
  → No additional cleanup needed
  → Automatic compliance
```

---

#### 💡 Why Combine RAFT + Crypto-Erasure?

These two features work together to provide **production-grade confidential computing**:

**RAFT Consensus:**
- 🔄 Keeps encryption keys synchronized across multiple KMS instances
- 🛡️ Ensures all instances agree on which keys have expired
- 📈 Enables horizontal scaling for high throughput
- 🔧 Provides automatic failover and recovery

**Crypto-Erasure:**
- ⏰ Automatically enforces data retention policies
- 🔒 Makes data permanently inaccessible after TTL
- ⚖️ Simplifies regulatory compliance (GDPR, HIPAA, etc.)
- 🛡️ Provides defense-in-depth security

**Together:**
```
Scalable + Secure + Compliant = Production-Ready System

┌────────────────────────────────────────────────┐
│  Multiple KMS Instances (RAFT Replicated)     │
│  ┌──────┐  ┌──────┐  ┌──────┐  ┌──────┐      │
│  │ KMS1 │  │ KMS2 │  │ KMS3 │  │ KMS4 │      │
│  └───┬──┘  └───┬──┘  └───┬──┘  └───┬──┘      │
│      │          │          │          │        │
│      └──────────┴──────────┴──────────┘        │
│              Synchronized State                │
│         (keys, expiration times)               │
└────────────────────────────────────────────────┘
                      ↓
          Every instance enforces TTL
          Keys expire automatically
          Compliance guaranteed!
```

---

#### 🚀 In This Workshop

While today's workshop uses a **single KMS instance** from a pool of 5, in production you would:

1. **Deploy a RAFT cluster** of 5-7 KMS instances
2. **Set appropriate TTLs** based on your data retention policies
3. **Monitor key expiration** to ensure compliance
4. **Scale horizontally** by adding more instances as load grows

**The workshop demonstrated:**
- ✅ Key rotation with TTL (Step 1: `ttl_seconds=7200`)
- ✅ Pipeline intermediate TTL (Step 6: `ttl_seconds=3600`)
- ✅ Automatic key expiration (happens in the background)

**Production additions:**
- RAFT replication across data centers
- Leader election and failover
- Distributed time synchronization
- Key expiration monitoring and alerts

---

### 🎓 Key Takeaway

**Confidential Federated Compute isn't just a research project - it's production-ready!**

With **RAFT consensus** and **crypto-erasure**, the KMS provides:
- 📈 **Scalability** to billions of requests
- 🛡️ **High availability** with automatic failover
- ⚖️ **Regulatory compliance** through automatic key expiration
- 🔒 **Defense in depth** with cryptographic data deletion

All while maintaining the **security guarantees** we explored today:
- TEE-based computation
- Hardware attestation
- Policy-bound encryption
- Cryptographic integrity

**This is the future of secure data processing!** 🎉

---

---

## 🎓 Key Takeaways

Congratulations on completing the Confidential Federated Compute workshop! Here's what you learned:

### 🔐 Core Concepts

1. **Trusted Execution Environments (TEEs)**
   - Hardware-enforced isolation for secure computation
   - Cryptographic attestation proves what code is running
   - Data never exposed in plaintext outside the TEE

2. **Key Management Service (KMS)**
   - Generates and manages encryption keys
   - Verifies TEE identity before sharing decryption keys
   - Enforces access policies through key derivation
   - Provides crypto-erasure for data TTL

3. **Access Policies**
   - Define who can access data (via reference values)
   - Specify data processing pipelines and transforms
   - Cryptographically bound to encryption (via hash)
   - Cannot be changed after data is encrypted

4. **HPKE Encryption**
   - Hybrid approach: public key + symmetric encryption
   - Uses X25519 for key exchange, AES-128-GCM for encryption
   - Policy hash is part of key derivation (HKDF "info" parameter)
   - Different policies → different keys → cryptographic integrity

### 🛡️ Security Properties

✅ **Confidentiality:** Data encrypted end-to-end, only TEEs can decrypt  
✅ **Integrity:** Policies cryptographically bound, tampering causes failure  
✅ **Attestation:** Hardware proves TEE identity before key release  
✅ **Auditability:** Policies published in transparency logs  
✅ **Crypto-erasure:** Keys auto-expire, data becomes inaccessible  

### 💡 Key Insights

1. **Cryptography enforces policy binding** - No complex validation needed, math does the work!
2. **Policy hash mismatch = safe failure** - Wrong key can't decrypt, system fails securely
3. **TEEs delegate trust** - Client only needs to trust KMS, not entire pipeline
4. **Attestation enables verification** - Hardware provides cryptographic proof
5. **Resource pooling improves efficiency** - 5 KMS instances serve 40 sessions

---

## 🚀 Next Steps

Want to learn more? Try these:

1. **Explore the KMS source code**
   - Located in: `containers/kms/`
   - Read the README.md for detailed architecture

2. **Run the full kms-flow-simulation**
   - More complex example with actual data processing
   - Located in: `containers/kms-flow-simulation/`

3. **Modify the policies**
   - Try creating multi-step pipelines
   - Experiment with different node configurations

4. **Test different scenarios**
   - Multiple clients uploading data
   - Concurrent pipeline executions
   - Key expiration and rotation

5. **Learn about Oak Project**
   - Visit: https://github.com/project-oak/oak
   - Explore more TEE-based applications

---

### Thank you for participating!

Questions? Found issues? Please reach out to the Confidential Federated Compute team.

**Remember:** This technology enables secure collaborative data analysis while preserving privacy. Use it responsibly! 🛡️

---